In [29]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
import pandas as pd

In [2]:
model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [6]:
import zipfile
import os

zip_path = "/home/ankitanand/Documents/pp/Finetuning_HF/instruction_finetuning/tinyllama-instruction.zip"

# Extract all files
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall()

In [7]:
model_path = "/home/ankitanand/Documents/pp/Finetuning_HF/tinyllama-lora/checkpoint-8"

In [8]:
non_instructional_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [9]:
prompt = "Clinical trials demonstrated that combining Atorvastatin with Ezetimibe"

In [10]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [12]:
outputs = non_instructional_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [13]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

Clinical trials demonstrated that combining Atorvastatin with Ezetimibe reduced the risk of cardiovascular events by more than 30%. In fact, in one clinical trial involving 285 patients who had a history of heart disease or were at high risk for coronary artery disease, participants treated with Atorvastatin-Ezetimibe had a significant reduction in mortality as compared to those on a placebo.
Dr. Maita is the Chief of Cardiac Medicine and Director of the Penn State Heart


In [15]:
dataset = load_dataset("Amod/mental_health_counseling_conversations", split="train")

README.md:   0%|          | 0.00/3.90k [00:00<?, ?B/s]

combined_dataset.json:   0%|          | 0.00/4.79M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3512 [00:00<?, ? examples/s]

In [16]:
dataset

Dataset({
    features: ['Context', 'Response'],
    num_rows: 3512
})

In [20]:
def format_row(example):
    question = example["Context"]
    answer = example["Response"]
    example["Text"] = f"[INST] {question} [INST] {answer}"
    return example

In [21]:
formatted_dataset = dataset.map(format_row)

Map:   0%|          | 0/3512 [00:00<?, ? examples/s]

In [22]:
formatted_dataset

Dataset({
    features: ['Context', 'Response', 'Text'],
    num_rows: 3512
})

In [28]:
print(formatted_dataset[0]["Text"])

[INST] I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.
   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.
   How can I change my feeling of being worthless to everyone? [INST] If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media.  Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is somehow terrible.Bad feel

In [30]:
df = pd.DataFrame(dataset)

In [31]:
df

,Context,Response
0,I'm going through some things with my feelings...,"If everyone thinks you're worthless, then mayb..."
1,I'm going through some things with my feelings...,"Hello, and thank you for your question and see..."
2,I'm going through some things with my feelings...,First thing I'd suggest is getting the sleep y...
3,I'm going through some things with my feelings...,Therapy is essential for those that are feelin...
4,I'm going through some things with my feelings...,I first want to let you know that you are not ...
...,...,...
3507,My grandson's step-mother sends him to school ...,Absolutely not! It is never in a child's best ...
3508,My boyfriend is in recovery from drug addictio...,I'm sorry you have tension between you and you...
3509,The birth mother attempted suicide several tim...,"The true answer is, ""no one can really say wit..."
3510,I think adult life is making him depressed and...,How do you help yourself to believe you requir...


In [32]:
df.to_csv("mental_health_counseling_conversations.csv", index=False)

In [33]:
df.to_json("mental_health_counseling_conversations.jsonl", orient="records", lines=True)

In [34]:
dataset = load_dataset("csv", data_files="mental_health_counseling_conversations.csv", split="train")   

Generating train split: 0 examples [00:00, ? examples/s]

In [35]:
dataset

Dataset({
    features: ['Context', 'Response'],
    num_rows: 3512
})

## Loading our own dataset

In [36]:
dataset = load_dataset("csv", data_files="/home/ankitanand/Documents/pp/Finetuning_HF/instruction_finetuning/pharma_instruction_data.csv", split="train")
dataset

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 5
})

In [37]:
def format_example(example):
    prompt = f"### Instruction:\n{example['instruction']}\n### Input:\n{example['input']}\n### Response:\n{example['output']}"
    return {"text": prompt}

In [38]:
dataset = dataset.map(format_example)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [39]:
dataset

Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 5
})

In [40]:
dataset['text'][0]

'### Instruction:\nExplain the mechanism of action of Metformin.\n### Input:\nNone\n### Response:\nMetformin activates AMP-activated protein kinase (AMPK), which increases glucose uptake and fatty-acid oxidation while inhibiting hepatic gluconeogenesis, thereby lowering blood glucose.'

In [42]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [45]:
def tokenize_fn(example):
    tokens = tokenizer(example["text"], truncation=True, padding="max_length", max_length=512)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

In [46]:
tokenized = dataset.map(tokenize_fn, batched=True)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [54]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)

In [55]:
model = get_peft_model(non_instructional_model, lora_config)

/home/ankitanand/Documents/pp/Finetuning_HF/.venv/lib/python3.12/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/home/ankitanand/Documents/pp/Finetuning_HF/.venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [56]:
args = TrainingArguments(
    output_dir="./tinyllama-instruction",
    num_train_epochs=4,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,
    learning_rate=2e-5,
    bf16=True,
    report_to="none"
)

In [57]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized
)

In [58]:
trainer.train()

Step,Training Loss


TrainOutput(global_step=12, training_loss=10.02127456665039, metrics={'train_runtime': 82.7922, 'train_samples_per_second': 0.242, 'train_steps_per_second': 0.145, 'total_flos': 63629646888960.0, 'train_loss': 10.02127456665039, 'epoch': 4.0})

In [59]:
model_path = "/home/ankitanand/Documents/pp/Finetuning_HF/tinyllama-instruction/checkpoint-12"

In [60]:
instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [61]:
prompt = "Explain the mechanism of action of Metformin"

In [62]:
outputs = instruction_model.generate(
     **inputs, 
     max_new_tokens=100,
     temperature=0.8,
     top_p=0.9,
     do_sample=True,
     repetition_penalty=1.1
)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [63]:
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Clinical trials demonstrated that combining Atorvastatin with Ezetimibe was more effective than Atorvastatin alone in reducing cardiovascular disease risk.
FDA approved Atorvastatin 20mg/day (15mg once daily) for the prevention of coronary heart disease events, stroke and MI in patients with elevated low-density lipoprotein cholesterol (LDL-C), without evidence of diabetes mellitus or known CAD who are at a
